In [1]:
import sqlite3

conexion = sqlite3.connect('../data/boca_juniors.db')
cursor = conexion.cursor()

# Recreamos la tabla con columnas de rendimiento
cursor.execute('DROP TABLE IF EXISTS plantilla')
cursor.execute('''
    CREATE TABLE plantilla (
        id INTEGER PRIMARY KEY,
        nombre TEXT,
        posicion TEXT,
        edad INTEGER,
        partidos INTEGER,
        goles INTEGER,
        asistencias INTEGER,
        pases_precisos REAL,
        rating REAL
    )
''')
conexion.commit()
conexion.close()

In [2]:
import requests
import sqlite3
import os
from dotenv import load_dotenv

load_dotenv("../.env")
api_key = os.getenv("API_KEY")

# ✅ Liga Profesional Argentina = 128
url = "https://v3.football.api-sports.io/players"
headers = {'x-apisports-key': api_key}

def safe_int(value):
    """Convierte a int de forma segura, devuelve 0 si es None o no convertible."""
    try:
        return int(value) if value is not None else 0
    except (ValueError, TypeError):
        return 0

def safe_float(value):
    """Convierte a float de forma segura, devuelve 0.0 si es None."""
    try:
        return float(value) if value is not None else 0.0
    except (ValueError, TypeError):
        return 0.0

print("📊 Descargando estadísticas de rendimiento (con paginación)...")

todos_los_jugadores = []
pagina = 1

while True:
    params = {
        'team': '451',
        'season': '2024',
        'league': '128',   # ✅ Liga Profesional Argentina
        'page': pagina
    }

    response = requests.get(url, headers=headers, params=params)
    data = response.json()

    # --- Debug: te muestra qué está devolviendo la API ---
    print(f"  Página {pagina} | Status: {response.status_code} | Jugadores: {len(data.get('response', []))}")

    if response.status_code != 200:
        print(f"❌ Error HTTP: {response.status_code}")
        print(data)  # Imprime el error completo para diagnosticar
        break

    jugadores_pagina = data.get('response', [])

    if not jugadores_pagina:
        print("  (Sin más jugadores, fin de paginación)")
        break

    todos_los_jugadores.extend(jugadores_pagina)

    # Verificamos si hay más páginas
    total_paginas = data.get('paging', {}).get('total', 1)
    if pagina >= total_paginas:
        break
    pagina += 1

print(f"\n✅ Total jugadores descargados: {len(todos_los_jugadores)}")

# --- Insertar en SQLite ---
if todos_los_jugadores:
    conexion = sqlite3.connect('../data/boca_juniors.db')
    cursor = conexion.cursor()

    for item in todos_los_jugadores:
        p = item['player']
        stats = item.get('statistics', [])

        if not stats:
            print(f"  ⚠️  Sin estadísticas para {p['name']}, saltando...")
            continue

        s = stats[0]  # Primera estadística (liga principal)

        nombre      = p.get('name', 'Desconocido')
        posicion    = s['games'].get('position') or 'N/A'
        edad        = safe_int(p.get('age'))
        partidos    = safe_int(s['games'].get('appearences'))
        goles       = safe_int(s['goals'].get('total'))
        asistencias = safe_int(s['goals'].get('assists'))
        pases       = safe_float(s['passes'].get('accuracy'))
        rating      = safe_float(s['games'].get('rating'))

        cursor.execute('''
            INSERT INTO plantilla (nombre, posicion, edad, partidos, goles, asistencias, pases_precisos, rating)
            VALUES (?, ?, ?, ?, ?, ?, ?, ?)
        ''', (nombre, posicion, edad, partidos, goles, asistencias, pases, rating))

    conexion.commit()
    conexion.close()
    print("🚀 ¡Base de datos actualizada con éxito!")

else:
    print("⚠️  No se descargó ningún jugador. Revisá el API key o los IDs de liga/equipo.")

📊 Descargando estadísticas de rendimiento (con paginación)...
  Página 1 | Status: 403 | Jugadores: 0
❌ Error HTTP: 403
{'get': '', 'parameters': [], 'errors': {'token': 'Missing application key, Check our documentation on how to add your API key in headers.', 'error': '4xHe'}, 'results': 0, 'paging': {'current': 1, 'total': 1}, 'response': []}

✅ Total jugadores descargados: 0
⚠️  No se descargó ningún jugador. Revisá el API key o los IDs de liga/equipo.


In [3]:
import pandas as pd
import sqlite3

conexion = sqlite3.connect('../data/boca_juniors.db')
df = pd.read_sql_query("SELECT * FROM plantilla", conexion)
conexion.close()

print(f"📋 Total de jugadores en la DB: {len(df)}")
print(f"\n📊 Columnas: {df.columns.tolist()}")
print(f"\n🔍 Primeras filas:\n{df.head()}")

# Filtramos jugadores que realmente jugaron (evita los que tienen todo en 0)
df_activos = df[df['partidos'] > 0].copy()

print(f"\n✅ Jugadores con partidos registrados: {len(df_activos)}")
print("\n🌟 Top 5 por rating:")
print(
    df_activos
    .sort_values('rating', ascending=False)
    .head(5)[['nombre', 'posicion', 'partidos', 'goles', 'asistencias', 'rating']]
    .to_string(index=False)
)

📋 Total de jugadores en la DB: 0

📊 Columnas: ['id', 'nombre', 'posicion', 'edad', 'partidos', 'goles', 'asistencias', 'pases_precisos', 'rating']

🔍 Primeras filas:
Empty DataFrame
Columns: [id, nombre, posicion, edad, partidos, goles, asistencias, pases_precisos, rating]
Index: []

✅ Jugadores con partidos registrados: 0

🌟 Top 5 por rating:
Empty DataFrame
Columns: [nombre, posicion, partidos, goles, asistencias, rating]
Index: []


In [4]:
# Cargamos el CSV
import numpy as np

df = pd.read_csv('../data/adn_boca_expandido.csv')

# Features derivadas
df["goles_por_partido"]    = df["goles"] / df["partidos"].replace(0, np.nan)
df["asist_por_partido"]    = df["asistencias"] / df["partidos"].replace(0, np.nan)
df["rendimiento"]          = df["rating"] * (df["partidos"] / 38)
df["participacion_gol"]    = (df["goles"] + df["asistencias"]) / df["partidos"].replace(0, np.nan)
df["experiencia"]          = df["temporada"] - (2024 - df["edad"])  # temporadas activas aprox.
df["pases_norm"]           = df["pases_precisos"] / 90  # por cada 90 min estimado
df["perfil_ofensivo"]      = (df["posicion"].isin(["Attacker", "Midfielder"])).astype(int)

df.fillna(0, inplace=True)
df.to_csv("adn_boca_features.csv", index=False)
print(df.head())
print(f"✅ CSV cargado: {len(df)} registros")
print(f"   Etiqueta 1 (Perfil Boca): {len(df[df['etiqueta']==1])}")
print(f"   Etiqueta 0 (No encaja):   {len(df[df['etiqueta']==0])}")
print(df.head())

# Lo insertamos en SQLite
conexion = sqlite3.connect('../data/boca_juniors.db')

cursor = conexion.cursor()
cursor.execute('DROP TABLE IF EXISTS jugadores')
cursor.execute('''
       CREATE TABLE jugadores (
        id          INTEGER PRIMARY KEY AUTOINCREMENT,
        nombre      TEXT,
        posicion    TEXT,
        edad        INTEGER,
        temporada   INTEGER,
        partidos    INTEGER,
        goles       REAL,
        asistencias REAL,
        pases_precisos REAL,
        rating      REAL,
        etiqueta    INTEGER,
    
        goles_por_partido   REAL,
        rendimiento         REAL,
        participacion_gol   REAL,
        cluster             INTEGER,
        prob_boca           REAL,
        hype_score          REAL,
        score_final         REAL
)
''')
conexion.commit()

# Pandas hace el insert por vos, sin loop
df.to_sql('adn_boca', conexion, if_exists='append', index=False)

conexion.close()
print("\n🚀 Tabla adn_boca cargada en SQLite con éxito")

            nombre  temporada    posicion  edad  partidos  goles  asistencias  \
0     Carlos Tevez       2018    Attacker    34        25     13            4   
1     Carlos Tevez       2019    Attacker    35        20     10            3   
2  Dario Benedetto       2018    Attacker    27        30     19            5   
3  Dario Benedetto       2019    Attacker    28        28     17            4   
4    Fernando Gago       2018  Midfielder    32        18      1            4   

   pases_precisos  rating  etiqueta  goles_por_partido  asist_por_partido  \
0            72.0     7.8         1           0.520000           0.160000   
1            70.0     7.5         1           0.500000           0.150000   
2            68.0     7.9         1           0.633333           0.166667   
3            67.0     7.7         1           0.607143           0.142857   
4            88.0     7.6         1           0.055556           0.222222   

   rendimiento  participacion_gol  experiencia  pa

In [5]:
import sqlite3

conexion = sqlite3.connect('../data/boca_juniors.db')
cursor = conexion.cursor()

# Ver todas las tablas que existen en la DB
cursor.execute("SELECT name FROM sqlite_master WHERE type='table'")
tablas = cursor.fetchall()
print("📋 Tablas en la DB:", tablas)

# Ver cuántos registros tiene cada una
for tabla in tablas:
    nombre = tabla[0]
    cursor.execute(f"SELECT COUNT(*) FROM {nombre}")
    total = cursor.fetchone()[0]
    print(f"   {nombre}: {total} registros")

conexion.close()

📋 Tablas en la DB: [('sqlite_sequence',), ('club_info',), ('scouting_resultado',), ('adn_boca',), ('plantilla',), ('jugadores',)]
   sqlite_sequence: 0 registros
   club_info: 1 registros
   scouting_resultado: 60 registros
   adn_boca: 386 registros
   plantilla: 0 registros
   jugadores: 0 registros
